# 03 - Abicart Analysis

## Purpose

The purpose of this notebook is to analyze the Abicart data formats used in the webshop import workflow.

The analysis focuses on:

- the existing Abicart product export
- the Abicart import sample format
- available fields and file structure
- data quality in the existing webshop export
- fields required for generating a valid future import file

## Imports

In [ ]:
from pathlib import Path
import pandas as pd

## File Configuration

In [ ]:
SUPPLIER = "snickers"

ABICART_EXPORT_PATH = (
    f"../data/{SUPPLIER}/abicart/abicart_full_export.csv"
)

ABICART_IMPORT_SAMPLE_PATH = (
    f"../data/{SUPPLIER}/abicart/import_exempelfil.csv"
)

## Abicart Export Analysis

### Raw File Inspection

Inspect the raw file before loading it into pandas or other tools. This helps identify:

- Encoding
- Delimiter
- Header rows
- File structure
- Multiline fields
- Unexpected formatting issues

In [ ]:
with open(ABICART_EXPORT_PATH, encoding="latin1") as f:
    for _ in range(5):
        print(repr(f.readline()))

### Load Dataset

In [ ]:
abicart_df = pd.read_csv(
    ABICART_EXPORT_PATH,
    encoding="latin1",
    skiprows=1,
    low_memory=False,
)

### Dataset Overview

In [ ]:
print("Rows:", len(abicart_df))
print("Columns:", len(abicart_df.columns))

abicart_df.head()

Observed:
- The full Abicart export contains 21,607 rows and 58 named columns.
- Product-level information is concentrated on the main product row.
- Additional rows for the same product may represent article-group, subgroup or customer-choice data rather than separate products.
- Historical variation attributes exist under several differently named colour and size columns.

### Schema Analysis

In [ ]:
abicart_df.info()

### Data Quality Checks

In [ ]:
abicart_df.isnull().sum()

In [ ]:
main_rows = abicart_df[abicart_df["Produkttyp"].notna()]

In [ ]:
duplicate_main_products = main_rows[
    main_rows["Produktnummer"].duplicated(keep=False)
].sort_values("Produktnummer")

duplicate_main_products[
    ["Produktnummer", "Produkttyp", "Namn (SV)", "Pris (SEK)"]
]

Observed:
- 1,191 rows contain product-level data, representing 1,187 unique product numbers.
- A small number of product numbers are reused across multiple product-level rows.
- These collisions originate from historical manual product management, including reused supplier identifiers and cases where pack/carton variants were entered under the same product number.
- Existing Abicart product numbers should therefore not automatically be treated as authoritative supplier identifiers during ETL matching.

#### Exact Duplicate Rows

In [ ]:
print("Duplicate rows:", abicart_df.duplicated().sum())

duplicate_rows = abicart_df[abicart_df.duplicated(keep=False)]
duplicate_rows.head(20)

Observed:
- The full export contains only 5 exact duplicate rows.
- The duplicate rows occur within products that also have historical duplicate or reused product numbers.
- Product `421120` is a confirmed duplicate product entry in the current Abicart catalogue.
- Historical manual product management has introduced a small number of data inconsistencies that must be considered during ETL matching.

### Product Structure Analysis

In [ ]:
print("Total rows:", len(abicart_df))
print("Rows with product-level data:", len(main_rows))
print("Unique product numbers:", main_rows["Produktnummer"].nunique())
print("Missing product names:", main_rows["Namn (SV)"].isna().sum())

Observed:
- The export contains 21,607 rows in total.
- 1,191 rows contain product-level data, identified by a populated `Produkttyp`.
- These rows represent 1,187 unique product numbers.
- The difference is explained by a small number of historical duplicate or reused product numbers identified during the data quality analysis.

In [ ]:
main_rows["Produkttyp"].value_counts(dropna=False)

Observed:
- 1,186 of the 1,191 product-level rows use the `Standard` product type.
- The catalogue also contains 4 `Giftcertificate` entries and 1 `Service` entry.
- `Standard` is therefore the relevant existing product type for normal physical products.

In [ ]:
standard_rows = main_rows[main_rows["Produkttyp"] == "Standard"]

print("Standard product rows:", len(standard_rows))
print(
    "Rows with customer-choice product number:",
    abicart_df["Produktnummer under kundens val"].notna().sum()
)

In [ ]:
abicart_df.loc[
    abicart_df["Produktnummer under kundens val"].notna(),
    [
        "Produktnummer",
        "Produktnummer under kundens val",
        "Pris (SEK)",
        "Färg (SV)",
        "Storlek (SV)",
    ],
].head(20)

Observed:
- `Produktnummer under kundens val` is widely used to store article numbers for customer-selectable variants.
- Historical coverage is incomplete because the field was populated manually and not consistently across all products.
- The existing catalogue therefore demonstrates the intended use of the field, but should not be used to infer complete variant coverage.

### Variant Article Number Strategy

For future supplier imports, variant-level article numbers should use the supplier's own article number whenever available.

For products structured by model, colour and size:
- the parent product represents the product/model,
- `Färg` and `Storlek` are used as customer-selectable variant attributes,
- `Produktnummer under kundens val` stores the supplier's exact variant article number.

Supplier article numbers should be transferred directly from the supplier data whenever available, rather than reconstructed manually.

In [ ]:
attribute_columns = [
    "FÄRG (SV)",
    "Färg (SV)",
    "färg (SV)",
    "STORLEK (SV)",
    "Storlek (SV)",
    "stl (SV)",
    "storlek Herr (SV)",
    "storlek (SV)",
]

abicart_df[attribute_columns].notna().sum()

Observed:
- Historical product data uses multiple attribute names for the same concepts of colour and size.
- Colour occurs as `FÄRG`, `Färg` and `färg`, while size occurs as `STORLEK`, `Storlek`, `stl` and `storlek`.
- This inconsistency originates from historical manual product management.
- Future automated imports should use standardized `Färg` and `Storlek` attributes consistently.

## Abicart Import Sample Analysis

### Raw File Inspection

In [ ]:
with open(ABICART_IMPORT_SAMPLE_PATH, encoding="utf-8") as f:
    for _ in range(5):
        print(repr(f.readline()))

### Load Import Sample

In [ ]:
abicart_import_sample = pd.read_csv(
    ABICART_IMPORT_SAMPLE_PATH,
    encoding="utf-8-sig"
)

print("Rows:", len(abicart_import_sample))
print("Columns:", len(abicart_import_sample.columns))

abicart_import_sample.columns.tolist()

Observed:
- The Abicart import sample contains 48 defined import fields.
- `Article number (required)` is explicitly marked as required.
- The import schema supports product content, pricing, images, EAN/SKU identifiers, categorization, SEO metadata, VAT, stock management and quantity rules.
- The import sample defines the available import structure but does not contain product data.
- The full webshop export contains 58 columns, including existing product data, customer-choice article numbers, product attributes and category hierarchy fields.

### Import Field Analysis

In [ ]:
for i, column in enumerate(abicart_import_sample.columns):
    print(f"{i}: {column}")

### Relevant Import Fields

The Abicart import template supports 48 fields, but the automated supplier workflow does not need to populate every available field.

For the planned supplier workflow, variant attributes such as `Färg` and `Storlek`, together with the supplier's exact variant article number, also need to be represented in the generated import structure.

- `Article number (required)`
- `Article type (new only)`
- `Hide article (1/0)`
- `Article name (one column per language)`
- `Description (one column per language)`
- `Name of article group (in your default language)`
- `Name of subgroup (in your default language)`
- `Price (one column per currency)`
- `Image (via HTTP or FTP)`
- `Weight`
- `EAN code`
- `Introductory text`
- `Purchase price (one column per currency)`
- `Moms percentage`
- stock-related fields where applicable
- quantity-related fields where applicable

### Customer Choice Fields

The standard Abicart import sample does not list customer-choice fields such as `Färg`, `Storlek` or `Produktnummer under kundens val`.

However, previous Snickers imports used these fields successfully as additional import columns.

For the automated supplier workflow:

- `Produktnummer` represents the parent product/model.
- `Produktnummer under kundens val` stores the supplier's exact variant article number.
- `Färg` represents the selectable colour.
- `Storlek` represents the selectable size.

These fields must therefore be included in the generated import structure in addition to the relevant standard Abicart import fields.

## Findings

- The full Abicart export contains 21,607 rows and 58 named columns.
- The export starts with `sep=,` followed by a header row containing the Abicart field names.
- Product descriptions may span multiple lines within quoted CSV fields.
- 1,191 rows contain product-level data, identified by a populated `Produkttyp`, representing 1,187 unique product numbers.
- All identified product-level rows contain a product name.
- `Standard` is the dominant product type, with 1,186 product-level rows. The catalogue also contains 4 `Giftcertificate` entries and 1 `Service` entry.
- Product-level rows contain the main product information, while additional rows may contain category, customer-choice or variant-related data.
- `Produktnummer under kundens val` is widely used for variant-level article numbers, but historical coverage is incomplete due to manual product management.
- Historical product data uses multiple attribute names for colour and size, including differences in capitalization and naming.
- Future automated imports should standardize customer-choice attributes to `Färg` and `Storlek`.
- Future variant-level article numbers should use the supplier's exact article number whenever available.
- Historical manual product management has resulted in a small number of reused product numbers, duplicate entries and malformed manually constructed variant identifiers.
- The full export contains 5 exact duplicate rows.
- The Abicart import sample contains 48 standard import fields and defines the import structure without containing product data.
- The planned supplier workflow also requires customer-choice fields such as `Färg`, `Storlek` and `Produktnummer under kundens val`.
- Historical Abicart data contains legacy inconsistencies that should not be used as the source of truth for future automated imports.

## Open Questions

- How should the additional customer-choice columns `Färg`, `Storlek` and `Produktnummer under kundens val` be mapped together with the standard Abicart import schema in the generated file?
- Which additional fields, beyond `Article number (required)`, are required in practice when creating new products through an Abicart import?
- How should category hierarchy beyond the standard article group and subgroup fields be represented in the generated import?
- Which existing Abicart products should be updated, replaced, hidden or left unchanged when supplier imports are introduced?